In [65]:
import numpy as np
import pandas as pd
import pickle
import celer
from datetime import datetime, timedelta
from typing import List, Dict, Tuple
from knockpy import KnockoffFilter
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def moving_average_smoother(signal, window_length = 7):
    signal_padded = np.append(np.nan * np.ones(window_length - 1), signal)
    signal_smoothed = (np.convolve(signal_padded, np.ones(window_length, dtype=int), mode="valid")/ window_length)
    return signal_smoothed

def standardize_X(X):
    X_mean, X_std = X.mean(axis=0), X.std(axis=0)
    X_std[X_std == 0] = 1
    standardized_X = (X - X_mean) / X_std
    standardized_X[np.isnan(standardized_X)] = 0
    return standardized_X, X_mean, X_std

def standardize_y(y):
    y_mean, y_std = y.mean(), y.std()
    standardized_y = (y - y_mean) / y_std
    standardized_y[np.isnan(standardized_y)] = 0
    return standardized_y, y_mean, y_std

def recover_coefficients(coef, X_std, y_std):
    return (coef / X_std) * y_std

def recover_intercept(X, y, coef, intercept_length):
    intercept = y - np.dot(X, coef)
    return np.mean(intercept[-intercept_length:])

def group_X(X, lag_set):
    grouped_X = np.array([X[r_idx - lag_set, :].T.flatten() for r_idx in range(max(lag_set), X.shape[0])])
    return grouped_X

def extend_indices(indices, group_size):
    extended_indices = []
    for idx in indices:
        extended_indices.extend(list(range(idx * group_size, idx * group_size + group_size)))
    return extended_indices

def shrink_indices(extended_indices, group_size):
    indices = [extended_indices[i] // group_size for i in range(0, len(extended_indices), group_size)]
    return indices

def date_before(date_int, diff_days):
    date_obj = datetime.strptime(str(date_int), '%Y%m%d')
    new_date_obj = date_obj - timedelta(days=int(diff_days))
    new_date_int = int(new_date_obj.strftime('%Y%m%d'))
    return new_date_int

def date_after(date_int, diff_days):
    date_obj = datetime.strptime(str(date_int), '%Y%m%d')
    new_date_obj = date_obj + timedelta(days=int(diff_days))
    new_date_int = int(new_date_obj.strftime('%Y%m%d'))
    return new_date_int

# HealthDataHandler(region_list, target_data, region_data_dict)
# both target_data and region_data_dict should be smoothed
class HealthDataHandler:
    def __init__(self, region_list: List[str], lag_set: np.ndarray, report_freq_days: int, target_data: pd.DataFrame, region_data: Dict[str, pd.DataFrame]):
        self.region_list = region_list
        self.lag_set = lag_set
        self.report_freq_days = report_freq_days
        self.target_data = target_data
        self.region_data = region_data

    def get_X_y(self, start_date: int, end_date: int, codes: List[str]):
        self.X = {}
        self.standardized_X = {}
        self.X_mean = {}
        self.X_std = {}
        self.y = {}
        self.standardized_y = {}
        self.y_mean = {}
        self.y_std = {}
        self.standardized_grouped_X = {}
        self.grouped_X = {}
        for region in self.region_list:
            X = self.region_data[region].loc[date_before(start_date, max(self.lag_set) * self.report_freq_days):end_date - self.report_freq_days][codes].to_numpy()
            y = self.target_data.loc[start_date:end_date - self.report_freq_days, state].to_numpy()
            standardized_X, X_mean, X_std = standardize_X(X)
            standardized_y, y_mean, y_std = standardize_y(y)
            standardized_grouped_X = group_X(standardized_X, lag_set = self.lag_set)
            grouped_X = group_X(X, lag_set = self.lag_set)
            X = X[-(len(y)):]
            standardized_X = standardized_X[-(len(y)):]
            self.X[region] = X
            self.standardized_X[region] = standardized_X
            self.X_mean[region] = X_mean
            self.X_std[region] = X_std
            self.y[region] = y
            self.standardized_y[region] = standardized_y
            self.y_mean[region] = y_mean
            self.y_std[region] = y_std
            self.standardized_grouped_X[region] = standardized_grouped_X
            self.grouped_X[region] = grouped_X
        self.nS = len(y)
        self.nG = X.shape[1]
        self.nV = grouped_X.shape[1]

    def run_all_single(self, intercept_length, model, alpha = 0.3, states_prev_g = None, maxsteps = 5, states_prev_W = None, tau = 0.5, M = 10, M_max = 30, fdr = 0.1):
        # grpLasso derandomKnock forward
        group_size = len(self.lag_set)
        self.active_g_indices = {}
        self.recovered_coef = {}
        self.recovered_intercept = {}
        if model == 'grpLasso':
            self.unit_coef = {}
            for region in self.region_list:
                if states_prev_g is None:
                    prev_g_indices = None
                else:
                    prev_g_indices = states_prev_g[region]
                active_g_indices, unit_coef, recovered_coef, recovered_intercept = SIS_grpLasso_KKT(self.X[region], self.y[region], self.standardized_grouped_X[region], self.standardized_y[region], self.grouped_X[region], self.X_std[region], self.y_std[region], group_size, intercept_length, alpha = alpha, prev_g_indices = prev_g_indices)
                # model_coef = unit_coef if coef_method == 'standardized' else recovered_coef
                self.active_g_indices[region] = active_g_indices
                self.recovered_coef[region] = recovered_coef
                self.recovered_intercept[region] = recovered_intercept
                self.unit_coef[region] = unit_coef
        elif model == 'forward':
            for region in self.region_list:
                active_g_indices, recovered_coef, recovered_intercept = forward_stepwise_regression(self.grouped_X[region], self.y[region], group_size, intercept_length, maxsteps = maxsteps)
                self.active_g_indices[region] = active_g_indices
                self.recovered_coef[region] = recovered_coef
                self.recovered_intercept[region] = recovered_intercept
        elif model == 'forward_backward':
            for region in self.region_list:
                if states_prev_g is None:
                    prev_g_indices = None
                else:
                    prev_g_indices = states_prev_g[region]
                active_g_indices, recovered_coef, recovered_intercept = forward_backward_stepwise_regression(self.grouped_X[region], self.y[region], group_size, intercept_length, maxsteps = maxsteps, prev_g_indices = prev_g_indices)
                self.active_g_indices[region] = active_g_indices
                self.recovered_coef[region] = recovered_coef
                self.recovered_intercept[region] = recovered_intercept
        elif model == 'derandomKnock':
            self.states_curr_W = {}
            for region in self.region_list:
                if states_prev_W is None:
                    full_prev_W = None
                else:
                    full_prev_W = states_prev_W[region]
                active_g_indices, recovered_coef, recovered_intercept, full_curr_W = derandomKnock(self.X[region], self.y[region], self.standardized_grouped_X[region], self.standardized_y[region], self.grouped_X[region], group_size, intercept_length, full_prev_W = full_prev_W, tau = tau, M = M, M_max = M_max, fdr = fdr, v = 1)
                self.active_g_indices[region] = active_g_indices
                self.recovered_coef[region] = recovered_coef
                self.recovered_intercept[region] = recovered_intercept
                self.states_curr_W[region] = full_curr_W
        else:
            raise ValueError("Invalid model provided. Use 'grpLasso', 'forward' or 'derandomKnock'.")

    def get_test(self, start_test: int, codes: List[str], period_length: int):
        self.y_test = {}
        self.grouped_X_test = {}
        end_test = date_after(start_test, period_length)
        for region in self.region_list:
            X = self.region_data[region].loc[date_before(start_test, max(self.lag_set) * self.report_freq_days):end_test - self.report_freq_days][codes].to_numpy()
            y = self.target_data.loc[start_test:end_test - self.report_freq_days, state].to_numpy()
            grouped_X = group_X(X, lag_set = self.lag_set)
            self.y_test[region] = y
            self.grouped_X_test[region] = grouped_X
        self.test_dates = self.target_data.loc[start_test:end_test - self.report_freq_days, state].index.to_numpy()

    def get_informative_states(self, primary_state, region_method, auxiliary_region_list = None, nStates = 5):
        # marginal_diff
        if region_method == 'marginal_diff':
            if auxiliary_region_list is None:
                auxiliary_region_list = [s for s in self.region_list if s != primary_state]
            if nStates is not None:
                self.informative_region_list = np.array(auxiliary_region_list)[self.sort_marginal_diff(primary_state, auxiliary_region_list)[:nStates]]
                return self.informative_region_list
            else:
                raise ValueError("nStates must not be None for 'marginal_diff' method.")
        # sibling
        elif region_method == 'sibling':
            for sublist in mega_list:
                if primary_state in sublist:
                    self.informative_region_list = [s for s in sublist if s != primary_state]
                    return self.informative_region_list
            raise ValueError(f"{primary_state} not found in any sibling groups.")
        else:
            raise ValueError("Invalid region_method provided. Use 'marginal_diff' or 'sibling'.")
    
    def sort_marginal_diff(self, primary_state, auxiliary_region_list):
        primary_Xty = np.dot(self.standardized_X[primary_state].T, self.standardized_y[primary_state]) / self.nS
        auxiliary_region_Xty = np.array([np.dot(self.standardized_X[auxiliary_state].T, self.standardized_y[auxiliary_state]) / self.nS for auxiliary_state in auxiliary_region_list])
        abs_diff = np.abs(auxiliary_region_Xty - primary_Xty)
        Rhat = []
        for s_idx, auxiliary_state in enumerate(auxiliary_region_list):
            abs_diff_s = abs_diff[s_idx]
            screened_largest_diff = abs_diff_s[np.argsort(abs_diff_s)[-int(self.nS/3):]]
            Rhat.append(np.sum(screened_largest_diff ** 2))
        return np.argsort(Rhat)
    
    def Q_aggregation(self, B, X_test, y_test, total_step = 10, selection = False):
        # y_test -= y_test.mean()
        y_test = y_test.reshape(-1, 1)
        XB = np.dot(X_test, B)
        # XB -= XB.mean(axis = 0)
        # if(selection){#select beta.hat with smallest prediction error
        if not selection: 
            # }else{#Q-aggregation
            # theta.hat<- exp(-colSums((y.test-X.test%*%B)^2)/2)
            # theta_hat = np.exp(-np.sum((y_test - np.dot(X_test, B)) ** 2, axis = 0) / 2)
            theta_hat = np.exp(-np.mean((y_test - XB) ** 2, axis = 0) / 2)
            # theta.hat=theta.hat/sum(theta.hat)
            theta_hat /= np.sum(theta_hat)
            # theta.old=theta.hat
            theta_old = theta_hat.copy()
            # beta<-as.numeric(B%*%theta.hat)
            beta = np.dot(B, theta_hat)
            # beta.ew<-beta
            beta_ew = beta.copy()
            # for(ss in 1:total.step){
            for ss in range(total_step):
                Xbeta = np.dot(X_test, beta)
                # Xbeta -= Xbeta.mean()
                # theta.hat<- exp(-colSums((y.test-X.test%*%B)^2)/2+colSums((as.vector(X.test%*%beta)-X.test%*%B)^2)/8)
                theta_hat = np.exp(-np.mean((y_test - XB) ** 2, axis = 0) / 2 + 
                                np.mean((Xbeta.reshape(-1, 1) - XB) ** 2, axis = 0) / 8)
                # theta.hat<-theta.hat/sum(theta.hat)
                theta_hat /= np.sum(theta_hat)
                # beta<- as.numeric(B%*%theta.hat*1/4+3/4*beta)
                beta = np.dot(B, theta_hat) * 1/4 + 3/4 * beta
                # if(sum(abs(theta.hat-theta.old))<10^(-3)){break}
                if np.sum(np.abs(theta_hat - theta_old)) < 1e-3:
                    break
                # theta.old=theta.hat
                theta_old = theta_hat.copy()
            # }
        if selection or np.isnan(theta_hat).any():
            # khat<-which.min(colSums((y.test-X.test%*%B)^2))
            # theta.hat<-rep(0, ncol(B))
            # theta.hat[khat] <- 1
            # beta=B[,khat]
            khat = np.argmin(np.mean((y_test - XB) ** 2, axis = 0))
            theta_hat = np.zeros(B.shape[1])
            theta_hat[khat] = 1
            beta = B[:, khat]
        return theta_hat, beta
    
    def run_all_agg(self, intercept_length, region_method, auxiliary_region_list = None, nStates = 5, total_step = 10, selection = False):
        group_size = len(self.lag_set)
        self.recovered_coef_agg = {}
        self.recovered_intercept_agg = {}
        for primary_state in self.region_list:
            self.get_informative_states(primary_state, region_method, auxiliary_region_list, nStates)
            B = np.zeros((len(self.informative_region_list) + 1, self.nV))
            B[0] = self.recovered_coef[primary_state] / self.y_std[primary_state] * np.repeat(self.X_std[primary_state], group_size)
            for s_idx, informative_state in enumerate(self.informative_region_list):
                B[s_idx + 1] = self.recovered_coef[informative_state] / self.y_std[primary_state] * np.repeat(self.X_std[primary_state], group_size)
            B = B.T
            X_test = (self.grouped_X_test[primary_state] - np.repeat(self.X_mean[primary_state], group_size)) / np.repeat(self.X_std[primary_state], group_size)
            y_test = (self.y_test[primary_state] - self.y_mean[primary_state]) / self.y_std[primary_state]
            theta_hat, beta = self.Q_aggregation(B, X_test, y_test, total_step, selection)
            self.recovered_coef_agg[primary_state] = recover_coefficients(beta, np.repeat(self.X_std[primary_state], group_size), self.y_std[primary_state])
            self.recovered_intercept[primary_state] = recover_intercept(self.grouped_X_test[primary_state], self.y_test[primary_state], self.recovered_coef[primary_state], intercept_length)
            self.recovered_intercept_agg[primary_state] = recover_intercept(self.grouped_X_test[primary_state], self.y_test[primary_state], self.recovered_coef_agg[primary_state], intercept_length)
### grpLasso
# SIS - grpLasso - KKT
# class celer.GroupLasso(groups=1, alpha=1.0, max_iter=100, max_epochs=50000, p0=10, verbose=0, tol=0.0001, prune=True, fit_intercept=True, weights=None, warm_start=False)
# (1 / (2 * n_samples)) * ||y - X w||^2_2 + alpha * \sum_g weights_g ||w_g||_2
def fit_grpLasso(standardized_grouped_X, standardized_y, grouped_X, y, X_std, y_std, group_size, intercept_length, alpha, weights = None):
    grpLasso_model = celer.GroupLasso(groups = group_size, alpha = alpha, tol = 1e-4, fit_intercept = False, weights = weights)
    grpLasso_model.fit(standardized_grouped_X, standardized_y)
    recovered_coef = recover_coefficients(grpLasso_model.coef_, np.repeat(X_std, group_size), y_std)
    recovered_intercept = recover_intercept(grouped_X, y, recovered_coef, intercept_length)
    return grpLasso_model, recovered_coef, recovered_intercept

def check_KKT(standardized_grouped_X, standardized_y, coef, g_indices, group_size, alpha, weights = None, tol = 1e-4):
    residual = standardized_y - np.dot(standardized_grouped_X, coef)
    check_stats = []
    for g_idx in range(0, standardized_grouped_X.shape[1], group_size):
        standardized_grouped_X_g = standardized_grouped_X[:, g_idx * group_size : g_idx * group_size + group_size]
        check_stat = np.linalg.norm(standardized_grouped_X_g.T.dot(residual), 2) / standardized_grouped_X_g.shape[0]
        check_stats.append(check_stat)
    check_stats = np.array(check_stats)
    if weights is None:
        weights = 1
    check_threshs = alpha * weights
    violated_g_indices = np.where(check_stats > check_threshs + tol)[0]
    violated_g_indices = list(set(violated_g_indices) - set(g_indices))
    return violated_g_indices

def SIS(X, y, nSel):
    correlations = np.abs(np.corrcoef(X.T, y)[:-1, -1])
    np.nan_to_num(correlations, copy=False)
    sorted_indices = np.argsort(correlations)[::-1]
    SIS_g_indices = sorted_indices[:nSel]
    return SIS_g_indices

def SIS_grpLasso_KKT(X, y, standardized_grouped_X, standardized_y, grouped_X, X_std, y_std, group_size, intercept_length, alpha, prev_g_indices = None, weights = None):
    if prev_g_indices is None or len(prev_g_indices) == 0:
        violated_g_indices = SIS(X, y, nSel = int(len(y) / group_size) - 1)
    else:
        violated_g_indices = prev_g_indices
    SIS_g_indices = np.array([], dtype = int)
    while len(violated_g_indices) > 0:
        SIS_g_indices = list(set(SIS_g_indices) | set(violated_g_indices))
        SIS_v_indices = extend_indices(SIS_g_indices, group_size)
        grpLasso_model, recovered_coef, recovered_intercept = fit_grpLasso(standardized_grouped_X[:, SIS_v_indices], standardized_y, grouped_X[:, SIS_v_indices], y, X_std[SIS_g_indices], y_std, group_size, intercept_length, alpha, weights)
        full_unit_coef = np.zeros(standardized_grouped_X.shape[1])
        full_unit_coef[SIS_v_indices] = grpLasso_model.coef_
        violated_g_indices = check_KKT(standardized_grouped_X, standardized_y, full_unit_coef, SIS_g_indices, group_size, alpha, weights)
    full_recovered_coef = np.zeros(standardized_grouped_X.shape[1])
    full_recovered_coef[SIS_v_indices] = recovered_coef
    active_v_indices = np.where(full_recovered_coef != 0)[0]
    active_g_indices = shrink_indices(active_v_indices, group_size)
    return active_g_indices, full_unit_coef, full_recovered_coef, recovered_intercept


### forward
def forward_stepwise_regression(grouped_X, y, group_size, intercept_length, maxsteps = 5):
    active_g_indices = []
    inactive_g_indices = list(range(int(grouped_X.shape[1] / group_size)))
    residual = y
    beta = None
    for step in range(maxsteps):
        best_g_idx = None
        best_corr = 0
        for g_idx in inactive_g_indices:
            corr = np.linalg.norm(np.corrcoef(grouped_X[:, g_idx * group_size : g_idx * group_size + group_size].T, residual)[:-1, -1], 2)
            if corr > best_corr:
                best_g_idx = g_idx
                best_corr = corr
        if best_g_idx is not None:
            active_g_indices.append(best_g_idx)
            inactive_g_indices.remove(best_g_idx)
            active_v_indices = extend_indices(active_g_indices, group_size)
            X_active = grouped_X[:, active_v_indices]
            # beta, _, _, _ = np.linalg.lstsq(X_active, y, rcond=None)
            extended_X_active = np.hstack([np.ones((grouped_X.shape[0], 1)), X_active])
            beta, _, _, _ = np.linalg.lstsq(extended_X_active, y, rcond=None)
            residual = y - extended_X_active @ beta
        else:
            break
    forward_coef = np.zeros(grouped_X.shape[1])
    if beta is not None:
        forward_coef[active_v_indices] = beta[1:]
    recovered_intercept = recover_intercept(grouped_X, y, forward_coef, intercept_length)
    return active_g_indices, forward_coef, recovered_intercept

### forward_backward
def forward_backward_stepwise_regression(grouped_X, y, group_size, intercept_length, maxsteps = 5, prev_g_indices = None):
    active_g_indices = []
    inactive_g_indices = list(range(int(grouped_X.shape[1] / group_size)))
    if prev_g_indices is None or len(prev_g_indices) == 0:
        residual = y
        beta = None
    else:
        for g_idx in prev_g_indices:
            active_g_indices.append(g_idx)
            inactive_g_indices.remove(g_idx)
        active_v_indices = extend_indices(active_g_indices, group_size)
        X_active = grouped_X[:, active_v_indices]
        extended_X_active = np.hstack([np.ones((grouped_X.shape[0], 1)), X_active])
        beta, _, _, _ = np.linalg.lstsq(extended_X_active, y, rcond=None)
        residual = y - extended_X_active @ beta

    # for step in range(maxsteps):
    while len(active_g_indices) < 2 * maxsteps:
        best_g_idx = None
        best_corr = 0
        for g_idx in inactive_g_indices:
            corr = np.linalg.norm(np.corrcoef(grouped_X[:, g_idx * group_size : g_idx * group_size + group_size].T, residual)[:-1, -1], 2)
            if corr > best_corr:
                best_g_idx = g_idx
                best_corr = corr
        if best_g_idx is not None:
            active_g_indices.append(best_g_idx)
            inactive_g_indices.remove(best_g_idx)
            active_v_indices = extend_indices(active_g_indices, group_size)
            X_active = grouped_X[:, active_v_indices]
            # beta, _, _, _ = np.linalg.lstsq(X_active, y, rcond=None)
            extended_X_active = np.hstack([np.ones((grouped_X.shape[0], 1)), X_active])
            beta, _, _, _ = np.linalg.lstsq(extended_X_active, y, rcond=None)
            residual = y - extended_X_active @ beta
        else:
            break

    while len(active_g_indices) > maxsteps:
        worst_g_idx = None
        smallest_improvement = np.inf
        for g_idx in active_g_indices:
            temp_active_g_indices = active_g_indices.copy()
            temp_active_g_indices.remove(g_idx)
            temp_active_v_indices = extend_indices(temp_active_g_indices, group_size)
            X_active_temp = grouped_X[:, temp_active_v_indices]
            extended_X_active_temp = np.hstack([np.ones((grouped_X.shape[0], 1)), X_active_temp])
            beta_temp, _, _, _ = np.linalg.lstsq(extended_X_active_temp, y, rcond=None)
            residual_temp = y - extended_X_active_temp @ beta_temp
            improvement = np.sum(residual_temp**2) - np.sum(residual**2)
            if improvement < smallest_improvement:
                smallest_improvement = improvement
                worst_g_idx = g_idx
        active_g_indices.remove(worst_g_idx)
        active_v_indices = extend_indices(active_g_indices, group_size)
        X_active = grouped_X[:, active_v_indices]
        extended_X_active = np.hstack([np.ones((grouped_X.shape[0], 1)), X_active])
        beta, _, _, _ = np.linalg.lstsq(extended_X_active, y, rcond=None)
        residual = y - extended_X_active @ beta
    
    stepwise_coef = np.zeros(grouped_X.shape[1])
    if beta is not None:
        stepwise_coef[active_v_indices] = beta[1:]
    recovered_intercept = recover_intercept(grouped_X, y, stepwise_coef, intercept_length)
    return active_g_indices, stepwise_coef, recovered_intercept

### derandomKnock
def derandomKnock_filter(standardized_grouped_X, standardized_y, group_size, prev_W = None, tau = 0.5, M = 10, M_max = 30, fdr = 0.1, v = 1):
    p = int(standardized_grouped_X.shape[1] / group_size)
    groups = np.repeat(np.arange(1, p + 1, 1), group_size)
    pi = np.zeros(p)
    curr_W = [] if prev_W is None else prev_W
    if len(curr_W) > M_max - M:
        curr_W = curr_W[-(M_max - M):]
    for m in range(M):
        kfilter = KnockoffFilter(ksampler = 'gaussian', fstat = 'lasso')
        kfilter.forward(standardized_grouped_X, standardized_y, groups = groups, fdr = fdr)
        W = kfilter.W
        curr_W.append(W)
    for W in curr_W:
        if np.sum(W < 0) < v:
            S = np.where(W > 0)[0] 
            pi[S] += 1
        else:
            order_w = np.argsort(np.abs(W))[::-1]
            sorted_w = W[order_w]
            negid = np.where(sorted_w < 0)[0]
            TT = negid[v - 1]
            S = np.where(sorted_w[:TT] > 0)[0]
            S = order_w[S]
            pi[S] += 1
    pi /= len(curr_W)
    S = np.where(pi >= min(tau, max(pi)))[0]
    return S, pi, curr_W

def derandomKnock_regression(grouped_X, y, S_g_indices, group_size):
    derandomKnock_model = LinearRegression()
    S_v_indices = extend_indices(S_g_indices, group_size)
    derandomKnock_model.fit(grouped_X[:, S_v_indices], y)
    derandomKnock_model_coef = np.zeros(grouped_X.shape[1])
    derandomKnock_model_coef[S_v_indices] = derandomKnock_model.coef_
    return derandomKnock_model_coef, derandomKnock_model.intercept_

def derandomKnock(X, y, standardized_grouped_X, standardized_y, grouped_X, group_size, intercept_length, full_prev_W = None, tau = 0.5, M = 10, M_max = 30, fdr = 0.1, v = 1):
    SIS_g_indices = SIS(X, y, nSel = int(len(y) / group_size) - 1)
    if np.unique(standardized_y).shape[0] > 2:
        SIS_v_indices = extend_indices(SIS_g_indices, group_size)
        if full_prev_W is None:
            prev_W = None
        else:
            prev_W = [W[SIS_g_indices] for W in full_prev_W]
        S, pi, curr_W = derandomKnock_filter(standardized_grouped_X[:, SIS_v_indices], standardized_y, group_size, prev_W, tau, M, M_max, fdr, v)
        S_g_indices = SIS_g_indices[S]
        full_curr_W = [np.zeros(X.shape[1]) for w_idx in range(len(curr_W))]
        for w_idx, W in enumerate(curr_W):
            full_curr_W[w_idx][SIS_g_indices] = W
    else:
        S_g_indices = SIS_g_indices
        full_curr_W = full_prev_W
    recovered_coef, _ = derandomKnock_regression(grouped_X, y, S_g_indices, group_size)
    recovered_intercept = recover_intercept(grouped_X, y, recovered_coef, intercept_length)
    return S_g_indices, recovered_coef, recovered_intercept, full_curr_W

In [90]:
dfs = []
all_valid_icd10_codes = []
threshold = 822 * 0.2
for region in ['R1_Boston', 'R2_NewYork', 'R3_Philadelphia', 'R4_Atlanta', 'R5_Chicago', 'R910_WestCoast', 'R678_CentralUSA']:
    df = pd.read_csv(f'../AllofUs/{region}.csv', index_col=0)
    df.index = pd.to_datetime(df.index)
    dfs.append(df)
    # Calculate the number of dates with counts > 0 for each ICD-10 code
    counts_per_code = (df > 0).sum()

    # Identify ICD-10 codes that meet the threshold
    valid_icd10_codes = set(counts_per_code[counts_per_code > threshold].index)
    
    # Append valid ICD-10 codes for this region to the list
    all_valid_icd10_codes.append(valid_icd10_codes)

# Find the intersection of all valid ICD-10 codes across all regions
intersection_valid_icd10_codes = set.intersection(*all_valid_icd10_codes)

# Now `intersection_valid_icd10_codes` will contain the ICD-10 codes that are valid across all regions
print(len(intersection_valid_icd10_codes))

361


In [92]:
region_list = ['R1_Boston', 'R2_NewYork', 'R3_Philadelphia', 'R4_Atlanta', 'R5_Chicago', 'R910_WestCoast', 'R678_CentralUSA']
lag_set = np.array([14, 7, 0])
report_freq_days = 1
lag_set = (lag_set / report_freq_days).astype(int)
method_list = ['grpLasso', 'forward', 'forward_backward', 'derandomKnock']

train_period = 420
validation_period = 42
intercept_length = validation_period
test_period = 42
step_size = 42

intersection_valid_icd10_codes = np.sort(list(intersection_valid_icd10_codes))
all_codes = intersection_valid_icd10_codes[intersection_valid_icd10_codes != 'U07']
target_data = pd.DataFrame()
region_data_dict = {}
for region in region_list:
    region_data = pd.read_csv(f'../AllofUs/{region}.csv', index_col = 0, header = 0)[intersection_valid_icd10_codes].astype(float).apply(moving_average_smoother).iloc[6:]
    region_data.index = pd.to_datetime(region_data.index).strftime('%Y%m%d').astype(int)
    region_data_dict[region] = region_data
    target_data[region] = region_data['U07']

train_start_date_list = []
validation_start_date_list = []
test_start_date_list = []
train_start_date = date_after(min(region_data.index), max(lag_set) * report_freq_days)

validation_start_date = date_after(train_start_date, train_period)
test_start_date = date_after(validation_start_date, validation_period)
while test_start_date < 20220101:
    train_start_date_list.append(train_start_date)
    validation_start_date_list.append(validation_start_date)
    test_start_date_list.append(test_start_date)
    train_start_date = date_after(train_start_date, step_size)
    validation_start_date = date_after(train_start_date, train_period)
    test_start_date = date_after(validation_start_date, validation_period)

data_handler = HealthDataHandler(region_list, lag_set, report_freq_days, target_data, region_data_dict)
for method in method_list:
    phe_cnt_dict = {}
    y_dict = {}
    pred_dict = {}
    pred_agg_dict = {}
    date_plot_list = []
    phe_cnt_dict_region = {}
    for region in region_list:
        phe_cnt_dict_region[region] = {}
        y_dict[region] = []
        pred_dict[region] = []
        pred_agg_dict[region] = []
    cnt_features = {}
    dist_windows = {}
    for region in region_list:
        cnt_features[region] = []
        dist_windows[region] = []
    
    states_prev_W = None
    states_prev_g = None
    # selection = False
    for d_idx in range(len(train_start_date_list)):
        data_handler.get_X_y(start_date = train_start_date_list[d_idx], end_date = validation_start_date_list[d_idx], codes = all_codes)
        data_handler.get_test(start_test = validation_start_date_list[d_idx], codes = all_codes, period_length = validation_period)
        print(train_start_date_list[d_idx], '-', test_start_date_list[d_idx])
        
        data_handler.run_all_single(intercept_length, method, alpha = 0.2, states_prev_g = states_prev_g, maxsteps = 4, states_prev_W = states_prev_W, tau = 0.5, M = 10, M_max = 30, fdr = 0.1)
        for region in region_list:
            for g in data_handler.active_g_indices[region]:
                if all_codes[g] in phe_cnt_dict_region[region]:
                    phe_cnt_dict_region[region][all_codes[g]] += 1
                else:
                    phe_cnt_dict_region[region][all_codes[g]] = 1
        
        
        for region in region_list:
            cnt_features[region].append(len(data_handler.active_g_indices[region]))
            if states_prev_g is not None:
                dist_windows[region].append(len(set(data_handler.active_g_indices[region]).symmetric_difference(set(states_prev_g[region]))))
        data_handler.run_all_agg(intercept_length, region_method = 'marginal_diff', auxiliary_region_list = None, nStates = 5, total_step = 10)
    
        for region in region_list:
            for g in data_handler.active_g_indices[region]:
                if all_codes[g] in phe_cnt_dict:
                    phe_cnt_dict[all_codes[g]] += 1
                else:
                    phe_cnt_dict[all_codes[g]] = 1
    
        for region in region_list:
            top_5 = sorted(phe_cnt_dict_region[region], key=phe_cnt_dict_region[region].get, reverse=True)[:5]
            print(region)
            for key in top_5:
                print(f"{key}: {phe_cnt_dict_region[region][key]}")
        top_5 = sorted(phe_cnt_dict, key=phe_cnt_dict.get, reverse=True)[:5]
        print('agg')
        for key in top_5:
            print(f"{key}: {phe_cnt_dict[key]}")
        
        if method == 'derandomKnock':
            states_prev_W = data_handler.states_curr_W
        states_prev_g = data_handler.active_g_indices

20200421 - 20210727
R1_Boston
A04: 1
B96: 1
F06: 1
F14: 1
F25: 1
R2_NewYork
F10: 1
F43: 1
F60: 1
J31: 1
O99: 1
R3_Philadelphia
B34: 1
F11: 1
F31: 1
J06: 1
J12: 1
R4_Atlanta
D68: 1
E08: 1
G62: 1
I50: 1
J12: 1
R5_Chicago
C77: 1
F14: 1
F20: 1
F25: 1
F31: 1
R910_WestCoast
B34: 1
F31: 1
F90: 1
J01: 1
J12: 1
R678_CentralUSA
C50: 1
D05: 1
F17: 1
J12: 1
K75: 1
agg
J12: 4
F31: 3
F14: 2
F25: 2
M86: 2
20200602 - 20210907
R1_Boston
B96: 2
F14: 2
F25: 2
G31: 2
J02: 2
R2_NewYork
F10: 2
F60: 2
J31: 2
O99: 2
S43: 2
R3_Philadelphia
B34: 2
J06: 2
J12: 2
J45: 2
L98: 2
R4_Atlanta
D68: 2
E08: 2
G62: 2
I50: 2
J12: 2
R5_Chicago
C77: 2
F14: 2
J30: 2
L02: 2
L03: 2
R910_WestCoast
B34: 2
F90: 2
J01: 2
J12: 2
M86: 2
R678_CentralUSA
C50: 2
D05: 2
F17: 2
J12: 2
K75: 2
agg
J12: 8
F14: 4
M86: 4
O26: 4
S83: 4
20200714 - 20211019
R1_Boston
B96: 3
F14: 3
F25: 3
G31: 3
J02: 3
R2_NewYork
F10: 3
F60: 3
S43: 3
S83: 3
S93: 3
R3_Philadelphia
B34: 3
J06: 3
J45: 3
J12: 2
L98: 2
R4_Atlanta
D68: 3
E08: 3
G62: 3
I50: 3
J12: 3
R5_C

SolverError: Solver 'SCS' failed. Try another solver, or solve with verbose=True for more information.

In [75]:
cnt_features

{'R1_Boston': [4],
 'R2_NewYork': [4],
 'R3_Philadelphia': [4],
 'R4_Atlanta': [4],
 'R5_Chicago': [4],
 'R910_WestCoast': [4],
 'R678_CentralUSA': [4]}

In [94]:
region_list = ['R1_Boston', 'R2_NewYork', 'R3_Philadelphia', 'R4_Atlanta', 'R5_Chicago', 'R910_WestCoast', 'R678_CentralUSA']
lag_set = np.array([14, 7, 0])
report_freq_days = 1
lag_set = (lag_set / report_freq_days).astype(int)
method_list = ['derandomKnock']

train_period = 140
validation_period = 42
intercept_length = validation_period
test_period = 42
step_size = 42

intersection_valid_icd10_codes = np.sort(list(intersection_valid_icd10_codes))
all_codes = intersection_valid_icd10_codes[intersection_valid_icd10_codes != 'U07']
target_data = pd.DataFrame()
region_data_dict = {}
for region in region_list:
    region_data = pd.read_csv(f'../AllofUs/{region}.csv', index_col = 0, header = 0)[intersection_valid_icd10_codes].astype(float).apply(moving_average_smoother).iloc[6:]
    region_data.index = pd.to_datetime(region_data.index).strftime('%Y%m%d').astype(int)
    region_data_dict[region] = region_data
    target_data[region] = region_data['U07']

train_start_date_list = []
validation_start_date_list = []
test_start_date_list = []
train_start_date = date_after(min(region_data.index), max(lag_set) * report_freq_days)

validation_start_date = date_after(train_start_date, train_period)
test_start_date = date_after(validation_start_date, validation_period)
while test_start_date < 20220101:
    train_start_date_list.append(train_start_date)
    validation_start_date_list.append(validation_start_date)
    test_start_date_list.append(test_start_date)
    train_start_date = date_after(train_start_date, step_size)
    validation_start_date = date_after(train_start_date, train_period)
    test_start_date = date_after(validation_start_date, validation_period)

data_handler = HealthDataHandler(region_list, lag_set, report_freq_days, target_data, region_data_dict)
for method in method_list:
    phe_cnt_dict = {}
    y_dict = {}
    pred_dict = {}
    pred_agg_dict = {}
    date_plot_list = []
    phe_cnt_dict_region = {}
    for region in region_list:
        phe_cnt_dict_region[region] = {}
        y_dict[region] = []
        pred_dict[region] = []
        pred_agg_dict[region] = []
    cnt_features = {}
    dist_windows = {}
    for region in region_list:
        cnt_features[region] = []
        dist_windows[region] = []
    
    states_prev_W = None
    states_prev_g = None
    # selection = False
    for d_idx in range(len(train_start_date_list)):
        data_handler.get_X_y(start_date = train_start_date_list[d_idx], end_date = validation_start_date_list[d_idx], codes = all_codes)
        data_handler.get_test(start_test = validation_start_date_list[d_idx], codes = all_codes, period_length = validation_period)
        print(train_start_date_list[d_idx], '-', test_start_date_list[d_idx])
        
        data_handler.run_all_single(intercept_length, method, alpha = 0.2, states_prev_g = states_prev_g, maxsteps = 4, states_prev_W = states_prev_W, tau = 0.5, M = 10, M_max = 30, fdr = 0.1)
        for region in region_list:
            for g in data_handler.active_g_indices[region]:
                if all_codes[g] in phe_cnt_dict_region[region]:
                    phe_cnt_dict_region[region][all_codes[g]] += 1
                else:
                    phe_cnt_dict_region[region][all_codes[g]] = 1
        
        
        for region in region_list:
            cnt_features[region].append(len(data_handler.active_g_indices[region]))
            if states_prev_g is not None:
                dist_windows[region].append(len(set(data_handler.active_g_indices[region]).symmetric_difference(set(states_prev_g[region]))))
        data_handler.run_all_agg(intercept_length, region_method = 'marginal_diff', auxiliary_region_list = None, nStates = 5, total_step = 10)
    
        for region in region_list:
            for g in data_handler.active_g_indices[region]:
                if all_codes[g] in phe_cnt_dict:
                    phe_cnt_dict[all_codes[g]] += 1
                else:
                    phe_cnt_dict[all_codes[g]] = 1
    
        for region in region_list:
            top_5 = sorted(phe_cnt_dict_region[region], key=phe_cnt_dict_region[region].get, reverse=True)[:5]
            print(region)
            for key in top_5:
                print(f"{key}: {phe_cnt_dict_region[region][key]}")
        top_5 = sorted(phe_cnt_dict, key=phe_cnt_dict.get, reverse=True)[:5]
        print('agg')
        for key in top_5:
            print(f"{key}: {phe_cnt_dict[key]}")
        
        if method == 'derandomKnock':
            states_prev_W = data_handler.states_curr_W
        states_prev_g = data_handler.active_g_indices

20200421 - 20201020
R1_Boston
G40: 1
R2_NewYork
Z72: 1
S09: 1
R3_Philadelphia
S52: 1
J12: 1
R4_Atlanta
R60: 1
R41: 1
L71: 1
R5_Chicago
R15: 1
C83: 1
R910_WestCoast
M96: 1
M67: 1
R678_CentralUSA
K29: 1
F12: 1
H10: 1
Z90: 1
D23: 1
agg
G40: 1
Z72: 1
S09: 1
S52: 1
J12: 1
20200602 - 20201201
R1_Boston
G40: 1
I73: 1
R2_NewYork
S09: 2
Z72: 1
R3_Philadelphia
S52: 1
J12: 1
S62: 1
R4_Atlanta
R60: 1
R41: 1
L71: 1
J12: 1
R5_Chicago
C83: 2
R15: 1
R910_WestCoast
M96: 2
M67: 1
L85: 1
R678_CentralUSA
F12: 2
Z90: 2
K29: 1
H10: 1
D23: 1
agg
S09: 2
J12: 2
C83: 2
M96: 2
F12: 2
20200714 - 20210112


/net/dali/home/mscbio/rul98/miniconda3/envs/struct/lib/python3.9/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/net/dali/home/mscbio/rul98/miniconda3/envs/struct/lib/python3.9/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


R1_Boston
G40: 1
I73: 1
M86: 1
R2_NewYork
S09: 2
Z72: 1
E10: 1
R3_Philadelphia
S52: 1
J12: 1
S62: 1
B34: 1
R4_Atlanta
J12: 2
R60: 1
R41: 1
L71: 1
R5_Chicago
C83: 2
R15: 1
H26: 1
S93: 1
R910_WestCoast
M96: 2
M67: 1
L85: 1
F90: 1
R678_CentralUSA
F12: 2
Z90: 2
K29: 1
H10: 1
D23: 1
agg
J12: 3
S09: 2
R15: 2
C83: 2
M96: 2
20200825 - 20210223


/net/dali/home/mscbio/rul98/miniconda3/envs/struct/lib/python3.9/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/net/dali/home/mscbio/rul98/miniconda3/envs/struct/lib/python3.9/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


R1_Boston
G40: 1
I73: 1
M86: 1
I50: 1
R2_NewYork
S09: 2
Z72: 1
E10: 1
I73: 1
D64: 1
R3_Philadelphia
B34: 2
S52: 1
J12: 1
S62: 1
R4_Atlanta
J12: 3
R60: 1
R41: 1
L71: 1
R5_Chicago
C83: 2
R15: 1
H26: 1
S93: 1
O26: 1
R910_WestCoast
M96: 2
M67: 1
L85: 1
F90: 1
J06: 1
R678_CentralUSA
F12: 2
Z90: 2
K29: 1
H10: 1
D23: 1
agg
J12: 4
S09: 2
R15: 2
C83: 2
M96: 2
20201006 - 20210406


/net/dali/home/mscbio/rul98/miniconda3/envs/struct/lib/python3.9/site-packages/numpy/lib/function_base.py:2853: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/net/dali/home/mscbio/rul98/miniconda3/envs/struct/lib/python3.9/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


R1_Boston
I50: 2
G40: 1
I73: 1
M86: 1
D61: 1
R2_NewYork
S09: 2
Z72: 1
E10: 1
I73: 1
D64: 1
R3_Philadelphia
B34: 3
S52: 1
J12: 1
S62: 1
B02: 1
R4_Atlanta
J12: 4
R60: 1
R41: 1
L71: 1
J96: 1
R5_Chicago
C83: 2
O26: 2
R15: 1
H26: 1
S93: 1
R910_WestCoast
M96: 2
J06: 2
M67: 1
L85: 1
F90: 1
R678_CentralUSA
F12: 2
Z90: 2
K29: 1
H10: 1
D23: 1
agg
J12: 6
B34: 3
S09: 2
R15: 2
C83: 2
20201117 - 20210518
R1_Boston
I50: 2
D61: 2
G40: 1
I73: 1
M86: 1
R2_NewYork
S09: 2
D64: 2
Z72: 1
E10: 1
I73: 1
R3_Philadelphia
B34: 4
B02: 2
S52: 1
J12: 1
S62: 1
R4_Atlanta
J12: 5
R60: 1
R41: 1
L71: 1
J96: 1
R5_Chicago
C83: 2
O26: 2
R15: 1
H26: 1
S93: 1
R910_WestCoast
J06: 3
M96: 2
M67: 1
L85: 1
F90: 1
R678_CentralUSA
F12: 2
Z90: 2
J12: 2
H91: 2
K29: 1
agg
J12: 9
B34: 5
J06: 3
S09: 2
R15: 2
20201229 - 20210629
R1_Boston
I50: 2
D61: 2
G40: 1
I73: 1
M86: 1
R2_NewYork
S09: 2
D64: 2
Z72: 1
E10: 1
I73: 1
R3_Philadelphia
B34: 5
B02: 2
S52: 1
J12: 1
S62: 1
R4_Atlanta
J12: 6
R60: 1
R41: 1
L71: 1
J96: 1
R5_Chicago
C83: 2
O26: 2

In [95]:
region_list = ['R1_Boston', 'R2_NewYork', 'R3_Philadelphia', 'R4_Atlanta', 'R5_Chicago', 'R910_WestCoast', 'R678_CentralUSA']
lag_set = np.array([14, 7, 0])
report_freq_days = 1
lag_set = (lag_set / report_freq_days).astype(int)
method_list = ['derandomKnock']

train_period = 350
validation_period = 42
intercept_length = validation_period
test_period = 42
step_size = 42

intersection_valid_icd10_codes = np.sort(list(intersection_valid_icd10_codes))
all_codes = intersection_valid_icd10_codes[intersection_valid_icd10_codes != 'U07']
target_data = pd.DataFrame()
region_data_dict = {}
for region in region_list:
    region_data = pd.read_csv(f'../AllofUs/{region}.csv', index_col = 0, header = 0)[intersection_valid_icd10_codes].astype(float).apply(moving_average_smoother).iloc[6:]
    region_data.index = pd.to_datetime(region_data.index).strftime('%Y%m%d').astype(int)
    region_data_dict[region] = region_data
    target_data[region] = region_data['U07']

train_start_date_list = []
validation_start_date_list = []
test_start_date_list = []
train_start_date = date_after(min(region_data.index), max(lag_set) * report_freq_days)

validation_start_date = date_after(train_start_date, train_period)
test_start_date = date_after(validation_start_date, validation_period)
while test_start_date < 20220101:
    train_start_date_list.append(train_start_date)
    validation_start_date_list.append(validation_start_date)
    test_start_date_list.append(test_start_date)
    train_start_date = date_after(train_start_date, step_size)
    validation_start_date = date_after(train_start_date, train_period)
    test_start_date = date_after(validation_start_date, validation_period)

data_handler = HealthDataHandler(region_list, lag_set, report_freq_days, target_data, region_data_dict)
for method in method_list:
    phe_cnt_dict = {}
    y_dict = {}
    pred_dict = {}
    pred_agg_dict = {}
    date_plot_list = []
    phe_cnt_dict_region = {}
    for region in region_list:
        phe_cnt_dict_region[region] = {}
        y_dict[region] = []
        pred_dict[region] = []
        pred_agg_dict[region] = []
    cnt_features = {}
    dist_windows = {}
    for region in region_list:
        cnt_features[region] = []
        dist_windows[region] = []
    
    states_prev_W = None
    states_prev_g = None
    # selection = False
    for d_idx in range(len(train_start_date_list)):
        data_handler.get_X_y(start_date = train_start_date_list[d_idx], end_date = validation_start_date_list[d_idx], codes = all_codes)
        data_handler.get_test(start_test = validation_start_date_list[d_idx], codes = all_codes, period_length = validation_period)
        print(train_start_date_list[d_idx], '-', test_start_date_list[d_idx])
        
        data_handler.run_all_single(intercept_length, method, alpha = 0.2, states_prev_g = states_prev_g, maxsteps = 4, states_prev_W = states_prev_W, tau = 0.5, M = 10, M_max = 30, fdr = 0.1)
        for region in region_list:
            for g in data_handler.active_g_indices[region]:
                if all_codes[g] in phe_cnt_dict_region[region]:
                    phe_cnt_dict_region[region][all_codes[g]] += 1
                else:
                    phe_cnt_dict_region[region][all_codes[g]] = 1
        
        
        for region in region_list:
            cnt_features[region].append(len(data_handler.active_g_indices[region]))
            if states_prev_g is not None:
                dist_windows[region].append(len(set(data_handler.active_g_indices[region]).symmetric_difference(set(states_prev_g[region]))))
        data_handler.run_all_agg(intercept_length, region_method = 'marginal_diff', auxiliary_region_list = None, nStates = 5, total_step = 10)
    
        for region in region_list:
            for g in data_handler.active_g_indices[region]:
                if all_codes[g] in phe_cnt_dict:
                    phe_cnt_dict[all_codes[g]] += 1
                else:
                    phe_cnt_dict[all_codes[g]] = 1
    
        for region in region_list:
            top_5 = sorted(phe_cnt_dict_region[region], key=phe_cnt_dict_region[region].get, reverse=True)[:5]
            print(region)
            for key in top_5:
                print(f"{key}: {phe_cnt_dict_region[region][key]}")
        top_5 = sorted(phe_cnt_dict, key=phe_cnt_dict.get, reverse=True)[:5]
        print('agg')
        for key in top_5:
            print(f"{key}: {phe_cnt_dict[key]}")
        
        if method == 'derandomKnock':
            states_prev_W = data_handler.states_curr_W
        states_prev_g = data_handler.active_g_indices

20200421 - 20210518
R1_Boston
I51: 1
B96: 1
S06: 1
R2_NewYork
M20: 1
L97: 1
F10: 1
S93: 1
R45: 1
R3_Philadelphia
J06: 1
D84: 1
R18: 1
R4_Atlanta
J12: 1
T84: 1
O99: 1
I77: 1
D68: 1
R5_Chicago
R68: 1
C77: 1
S92: 1
K80: 1
M16: 1
R910_WestCoast
J12: 1
R07: 1
J06: 1
E86: 1
R678_CentralUSA
J12: 1
K75: 1
D05: 1
T84: 1
I21: 1
agg
J12: 3
J06: 2
T84: 2
I51: 1
B96: 1
20200602 - 20210629
R1_Boston
I51: 2
B96: 2
S06: 1
R2_NewYork
R16: 2
F60: 2
M20: 1
L97: 1
F10: 1
R3_Philadelphia
J06: 1
D84: 1
R18: 1
B34: 1
R4_Atlanta
J12: 2
T84: 2
O99: 2
D68: 2
J96: 2
R5_Chicago
R68: 2
C77: 2
S92: 2
M16: 2
K80: 1
R910_WestCoast
J12: 2
R07: 2
J06: 2
E86: 2
S52: 1
R678_CentralUSA
K75: 2
D05: 2
T84: 2
N32: 2
L71: 2
agg
J12: 5
T84: 4
J06: 3
I51: 2
B96: 2
20200714 - 20210810
R1_Boston
B96: 3
I51: 2
S06: 1
R2_NewYork
R16: 3
F60: 3
L97: 2
S93: 2
M20: 1
R3_Philadelphia
J06: 2
B34: 2
D84: 1
R18: 1
L91: 1
R4_Atlanta
J12: 3
T84: 3
D68: 3
J96: 3
K50: 3
R5_Chicago
R68: 3
C77: 3
S92: 3
M16: 3
K80: 1
R910_WestCoast
J12: 3
J06: 3